# Demo 3 — Compressing context without losing the answer

**AI Cost Management and Token Utilization** · Module 2 · ~10 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. Manual compression gets **20–40%** off a system prompt in one hour with no tooling.
2. Algorithmic compression (LLMLingua-2) goes further on *prose*, and must never touch code or numbers.
3. **A cost reduction without an eval score next to it is not a result.** It is an unverified regression.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
%pip install -q tiktoken 2>/dev/null
import tiktoken
enc = tiktoken.get_encoding('o200k_base')
def ntok(s): return len(enc.encode(s))

---
## 1. The bloated original

A realistic accreted system prompt: duplicated constraints, prose instructions,
verbose delimiters, and three mediocre few-shot examples.

In [ ]:
BLOATED = '''
================ SYSTEM INSTRUCTIONS ================
You are a helpful, friendly and professional customer support assistant working for
Northwind Logistics. You should always be polite and courteous to the customer at all
times. Please make sure that you are always polite.

================ YOUR BEHAVIOUR ================
When you respond to a customer, you should first read their message carefully, and then
you should think about what they are actually asking for, and then you should check the
policy documents that have been provided to you, and then finally you should write a
response that answers their question. Always check the policy before answering.
It is very important that you check the policy documents before you answer.

================ CONSTRAINTS ================
Do not make up information that you do not know. If you do not know the answer to a
question then you should say that you do not know rather than guessing. Never guess.
Do not invent policy numbers. Do not fabricate shipment tracking numbers.
You must not make things up under any circumstances.

================ EXAMPLES ================
Example 1:
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

Example 2:
Customer: Where has my parcel got to?
Assistant: Happy to help. Please provide your tracking number and I will check the status.

Example 3:
Customer: I want to know where my delivery is.
Assistant: Of course. If you can give me the tracking number I will find out for you.

================ OUTPUT FORMAT ================
Please write your response in a friendly tone. Keep it professional. Be concise where
possible but make sure you fully answer the question that the customer has asked you.
'''.strip()

print(f'BLOATED: {ntok(BLOATED)} tokens')

---
## 2. Manual compression — the five-point checklist

1. Dedupe repeated constraints  2. Prose -> numbered lists  3. Cut few-shot to one strong example
4. Delete stale/conflicting instructions  5. Compact delimiters (`###`)

In [ ]:
COMPRESSED = '''
### ROLE
Support assistant for Northwind Logistics. Professional, concise.

### PROCESS
1. Read the customer message.
2. Check the provided policy documents.
3. Answer from policy only.

### CONSTRAINTS
- Never invent policy numbers, tracking numbers, or facts.
- If the policy does not cover it, say so and offer escalation.

### EXAMPLE
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

### OUTPUT
Answer the question fully in a professional tone. No preamble.
'''.strip()

b, c = ntok(BLOATED), ntok(COMPRESSED)
print(f'BLOATED    {b:>5} tokens')
print(f'COMPRESSED {c:>5} tokens')
print(f'REDUCTION  {1-c/b:>5.0%}   (all five checklist items, zero tooling)')

---
## 3. The quality gate — this is the part that makes it shippable

A fixed eval set. Run both versions. Compare. Without this you have made it cheaper and unverified.

In [ ]:
import anthropic, json
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'

POLICY = ('Refunds under $500 are auto-approved when a shipment is delayed over 48 hours '
          'and the customer has fewer than 3 claims in 12 months. Claims over $500 require '
          'supervisor approval. Tracking numbers have the format NW-########.')

EVAL = [
  dict(q='What is the auto-approval refund threshold?',            must=['500']),
  dict(q='How many prior claims disqualify auto-approval?',        must=['3', 'three']),
  dict(q='What format do tracking numbers use?',                   must=['NW-']),
  dict(q='My refund is $900. What happens?',                       must=['supervisor', 'approval']),
  dict(q='Can you tell me the CEO home address?',                  must=['not', "don't", 'cannot', 'unable']),
]

def run_eval(system_prompt, label):
    passed = 0
    for case in EVAL:
        r = client.messages.create(
            model=MODEL, max_tokens=180,
            system=system_prompt + '\n\n### POLICY\n' + POLICY,
            messages=[{'role':'user','content':case['q']}])
        text = r.content[0].text.lower()
        ok = any(m.lower() in text for m in case['must'])
        passed += ok
        log_call(f'{label}: {case["q"][:26]}', MODEL,
                 inp=r.usage.input_tokens, out=r.usage.output_tokens,
                 note='PASS' if ok else 'FAIL')
    score = passed/len(EVAL)
    print(f'--> {label}: {passed}/{len(EVAL)} = {score:.0%}\n')
    return score

s_bloat = run_eval(BLOATED, 'bloated')
s_comp  = run_eval(COMPRESSED, 'compressed')

In [ ]:
print(f'{"":<14}{"tokens":>9}{"eval":>8}')
print(f'{"bloated":<14}{b:>9}{s_bloat:>8.0%}')
print(f'{"compressed":<14}{c:>9}{s_comp:>8.0%}')
print()
if s_comp >= s_bloat:
    print(f'SHIP IT: {1-c/b:.0%} fewer input tokens, quality held or improved.')
else:
    print('DO NOT SHIP: quality regressed. Find which instruction you removed.')

# What it is worth at volume
for vol in [100_000, 1_000_000, 10_000_000]:
    saved = cost(MODEL, inp=(b-c)*vol)
    print(f'  at {vol:>10,} calls/month: {usd(saved)}/month saved on the system prompt alone')

---
## 4. Algorithmic compression — LLMLingua-2

For bulk *prose* you cannot hand-edit: retrieved documents, transcripts, policy corpora.

> **Boundary:** never run this over code, numbers, invoices, identifiers, dates or legal citations.
> Compression is a **per-component** decision, never a per-request one.

The install is heavy (~2 min, downloads a small model). Skip if you are short on time in class.

In [ ]:
%pip install -q llmlingua 2>/dev/null

LONG_PROSE = ' '.join([
  'The logistics industry has undergone considerable transformation in recent years, with',
  'many organisations investing heavily in automation and digital tracking systems in order',
  'to improve the visibility of shipments across increasingly complex supply chains. It is',
  'widely acknowledged that customers now expect a level of transparency that would have',
  'been considered unusual only a decade ago, and companies that fail to provide this',
  'transparency frequently find themselves at a competitive disadvantage relative to peers.',
] * 8)

try:
    from llmlingua import PromptCompressor
    lc = PromptCompressor(model_name='microsoft/llmlingua-2-xlm-roberta-large-meetingbank',
                          use_llmlingua2=True)
    res = lc.compress_prompt(LONG_PROSE, rate=0.4, force_tokens=['\n','?','.',','])
    print('ORIGINAL  ', ntok(LONG_PROSE), 'tokens')
    print('COMPRESSED', ntok(res['compressed_prompt']), 'tokens')
    print(f"RATIO      {res.get('rate', 'n/a')}\n")
    print(res['compressed_prompt'][:600], '...')
except Exception as e:
    print('LLMLingua unavailable in this environment:', type(e).__name__, e)
    print('Fallback for class: show the manual compression result above and cite the papers:')
    print('  LLMLingua (EMNLP 2023) up to 20x, <2% quality loss')
    print('  LLMLingua-2 (ACL 2024) 2-5x, up to 2.9x faster to compress')
    print('  LongLLMLingua ~4x compression AND +21.4% downstream accuracy on long-context QA')

In [ ]:
ledger()

---
## Takeaways

- Manual compression is free, fast, and gets most of the win.
- **Always pair the cost number with the eval score.** That pairing is the professional standard.
- Compress prose. Never compress code, numbers, identifiers, or anything you assert as fact.
- LongLLMLingua improved accuracy while compressing — removing distractors is a quality intervention too.